# Direct Trade Booking Via RabbitMQ

This notebook behaves like an external blotter or booking system. It bypasses `helix-rest`, books a trade directly into the Helix SQLite store through the `helix_runtime` client API, and publishes the RabbitMQ tasks that drive runtime processing.

Prerequisites:

- the Helix stack is already running
- open this notebook through `./scripts/linux/notebook_start.sh` or `./scripts/win/notebook_start.ps1`
- the selected `portfolio_id`, `instrument_id`, and `book` already exist in seeded reference data


In [1]:
from __future__ import annotations

import os
import sqlite3
from datetime import timedelta

from helix_runtime import TradeBookingRequest, book_trade, default_db_path, utc_now

In [2]:
DB_PATH = default_db_path()
print(f"DB_PATH={DB_PATH}")
print(f"HELIX_RABBITMQ_HOST={os.environ.get('HELIX_RABBITMQ_HOST', 'localhost')}")
print(f"HELIX_RABBITMQ_PORT={os.environ.get('HELIX_RABBITMQ_PORT', '5672')}")

DB_PATH=/Users/alexandershubert/git/helix/helix-store/helix.db
HELIX_RABBITMQ_HOST=localhost
HELIX_RABBITMQ_PORT=5672


In [17]:
import random

In [43]:
request = TradeBookingRequest(
    portfolio_id="PF-EQ",
    instrument_id="AAPL",
    side='SELL' if random.random() <= 0.5 else 'BUY',
    quantity=random.randint(0, 100),
    price=random.randint(180,200),
    settlement_date=(utc_now() + timedelta(days=1)).date().isoformat(),
    book="EQ-789",
)

request

booked = book_trade(request)
booked

BookedTrade(trade_id='TRD-PF-EQ-20260323085556576', position_id='PF-EQ-POS-20260323085556576', portfolio_id='PF-EQ', requested_at=datetime.datetime(2026, 3, 23, 8, 55, 56, 576000, tzinfo=datetime.timezone.utc))

In [12]:
with sqlite3.connect(DB_PATH) as conn:
    conn.row_factory = sqlite3.Row

    trade_row = conn.execute(
        "SELECT trade_id, portfolio_id, instrument_id, side, quantity, price, notional, status, updated_at FROM trades WHERE trade_id = ?",
        (booked.trade_id,),
    ).fetchone()

    position_rows = conn.execute(
        """
        SELECT instrument_id, quantity, direction, realized_pnl, unrealized_pnl, total_pnl, as_of_ts
        FROM position
        WHERE portfolio_id = ?
        ORDER BY as_of_ts DESC, instrument_id ASC
        LIMIT 10
        """,
        (booked.portfolio_id,),
    ).fetchall()

    pnl_row = conn.execute(
        "SELECT total_pnl, realized_pnl, unrealized_pnl, valuation_ts FROM pnl WHERE portfolio_id = ? ORDER BY valuation_ts DESC LIMIT 1",
        (booked.portfolio_id,),
    ).fetchone()

    risk_row = conn.execute(
        "SELECT delta, gross_exposure, net_exposure, var_95, valuation_ts FROM risk WHERE portfolio_id = ? ORDER BY valuation_ts DESC LIMIT 1",
        (booked.portfolio_id,),
    ).fetchone()

display(dict(trade_row) if trade_row else None)
display([dict(row) for row in position_rows])
display(dict(pnl_row) if pnl_row else None)
display(dict(risk_row) if risk_row else None)

{'trade_id': 'TRD-PF-EQ-20260322124750312',
 'portfolio_id': 'PF-EQ',
 'instrument_id': 'AAPL',
 'side': 'SELL',
 'quantity': 175.0,
 'price': 60.5,
 'notional': 10587.5,
 'status': 'processed',
 'updated_at': '2026-03-22T12:47:50.312179Z'}

[{'instrument_id': 'AAPL',
  'quantity': 125.0,
  'direction': 'LONG',
  'realized_pnl': 729.17,
  'unrealized_pnl': 16958.33,
  'total_pnl': 17687.5,
  'as_of_ts': '2026-03-22T12:47:50.312179Z'},
 {'instrument_id': 'AAPL',
  'quantity': 300.0,
  'direction': 'LONG',
  'realized_pnl': 0.0,
  'unrealized_pnl': 40700.0,
  'total_pnl': 40700.0,
  'as_of_ts': '2026-03-22T12:47:40.504824Z'},
 {'instrument_id': 'AAPL',
  'quantity': 125.0,
  'direction': 'LONG',
  'realized_pnl': 0.0,
  'unrealized_pnl': 17687.5,
  'total_pnl': 17687.5,
  'as_of_ts': '2026-03-22T12:47:10.183222Z'}]

{'total_pnl': 17687.5,
 'realized_pnl': 729.17,
 'unrealized_pnl': 16958.33,
 'valuation_ts': '2026-03-22T12:47:50.312179Z'}

{'delta': 24000.0,
 'gross_exposure': 24000.0,
 'net_exposure': 24000.0,
 'var_95': 9900.0,
 'valuation_ts': '2026-03-22T12:47:50.312179Z'}